In [ ]:
library(monocle3)
library(Seurat)
library(SeuratWrappers)
library(SingleCellExperiment)
library(magrittr)
library(dplyr)
library(ggplot2)
library(patchwork)

options(monocle3.irlba = FALSE)


In [ ]:
getwd()

In [ ]:
# Project configuration. Override these values for your local data layout.
project_ID <- "pool_Myeloid"
mtx <- "transcriptome"
n_neighbors <- 80
min_dist <- 0.4
early_genes <- c("Mafb", "C1qb", "C1qa", "C1qc")
set.seed(0)


In [ ]:
outputdir <- file.path(mtx, project_ID)
deg_dir <- paste0(outputdir, "_DEG")
input_rds <- file.path(deg_dir, paste0(project_ID, "_anno_raw.rds"))
dir.create(deg_dir, recursive = TRUE, showWarnings = FALSE)

if (!file.exists(input_rds)) {
  stop("Input Seurat object not found: ", input_rds)
}


In [ ]:
seurat <- readRDS(input_rds)
required_metadata <- c("Sample_id", "treatment", "anno_sub")
missing_metadata <- setdiff(required_metadata, colnames(seurat[[]]))
if (length(missing_metadata) > 0) {
  stop("Missing Seurat metadata columns: ", paste(missing_metadata, collapse = ", "))
}


In [ ]:
table(seurat$Sample_id)

In [ ]:
table(seurat$treatment)

In [ ]:
seurat.list <- SplitObject(seurat, split.by = "Sample_id")
for (i in 1:length(seurat.list)) {
  seurat.list[[i]] <- SCTransform(seurat.list[[i]], verbose = FALSE)
}

In [ ]:
if (length(seurat.list) < 2) {
  stop("At least two Sample_id groups are required for integration.")
}


In [ ]:
# 使用 Seurat integration 做批次整合
features <- SelectIntegrationFeatures(object.list = seurat.list, nfeatures = 3000)
seurat.list <- PrepSCTIntegration(object.list = seurat.list, anchor.features = features)
anchors <- FindIntegrationAnchors(object.list = seurat.list, normalization.method = "SCT", anchor.features = features)

In [ ]:
seurat.integrated <- IntegrateData(anchorset = anchors, normalization.method = "SCT")

In [ ]:
# PCA must be computed before UMAP when dims is used.
seurat.integrated <- RunPCA(seurat.integrated, npcs = 30, verbose = FALSE)
seurat.integrated <- RunUMAP(seurat.integrated, dims = 1:30, seed.use = 0)


In [ ]:
FeaturePlot(seurat.integrated, 
            features = "Cd40",
            label = TRUE, reduction = "umap",
            label.size = 4)

In [ ]:
VlnPlot(
  seurat.integrated,
  features = "Cd40",
  group.by = "treatment",
  pt.size = 0
)


In [ ]:
trajectory_assay <- if ("RNA" %in% Assays(seurat.integrated)) "RNA" else DefaultAssay(seurat.integrated)
DefaultAssay(seurat.integrated) <- trajectory_assay
cds <- as.cell_data_set(seurat.integrated)


In [ ]:
if (!"anno_sub" %in% colnames(colData(cds))) {
  stop("anno_sub was not transferred to the Monocle3 cell_data_set.")
}


In [ ]:
# Build the Monocle3 PCA representation from the expression assay before alignment.
cds <- preprocess_cds(cds, num_dim = 30, method = "PCA")


In [ ]:
# 在降维时回归 batch
cds <- align_cds(cds, alignment_group = "Sample_id")



In [ ]:
# Recalculate the Monocle3 embedding after batch alignment and then cluster cells.
cds <- reduce_dimension(
  cds,
  reduction_method = "UMAP",
  umap.n_neighbors = n_neighbors,
  umap.min_dist = min_dist
)
cds <- cluster_cells(cds, resolution = 0.001)


In [ ]:
select_root_by_genes <- function(cds, early_genes, n_cells = 50) {
  present_genes <- intersect(early_genes, rownames(cds))
  if (length(present_genes) == 0) {
    stop("None of the configured early_genes are present in the dataset.")
  }

  gene_expr <- SummarizedExperiment::assay(cds, "counts")[present_genes, , drop = FALSE]
  avg_expression <- Matrix::colMeans(gene_expr)
  n_cells <- min(as.integer(n_cells), length(avg_expression))
  names(sort(avg_expression, decreasing = TRUE))[seq_len(n_cells)]
}


In [ ]:
# Monocle3 requires a learned principal graph before cells can be ordered.
cds <- learn_graph(
  cds,
  learn_graph_control = list(ncenter = 20, minimal_branch_len = 1),
  use_partition = TRUE,
  close_loop = FALSE
)
root_cells <- select_root_by_genes(cds, early_genes, n_cells = 50)
cds <- order_cells(cds, root_cells = root_cells)


In [ ]:
plot_cells(cds, label_groups_by_cluster = TRUE, label_leaves = FALSE, 
           label_branch_points = FALSE,label_principal_points=FALSE,
           group_label_size = 5) +
  theme(legend.position = "right") +
  guides(color = guide_legend(title = "Cell Type", override.aes = list(size = 4)))

In [ ]:
plot_cells(cds, color_cells_by = "anno_sub",label_groups_by_cluster = TRUE, label_leaves = FALSE, 
           label_branch_points = FALSE,label_principal_points=FALSE,
           group_label_size = 5) +
  theme(legend.position = "right") +
  guides(color = guide_legend(title = "anno_sub", override.aes = list(size = 4)))

In [ ]:
p1 <- plot_cells(cds, 
                     color_cells_by = "anno_sub",
                     label_groups_by_cluster = FALSE,
                     label_leaves = FALSE, 
                     label_branch_points = FALSE,             
                     group_label_size = 5,cell_size=1.) +
        theme(legend.position = "right") +
        guides(color = guide_legend(title = "Cell Type", override.aes = list(size = 3))) +
        ggtitle(paste("n_neighbors =", n_neighbors, "min_dist =", min_dist))
p2 <- plot_cells(cds, 
                     label_groups_by_cluster = TRUE,
                     label_leaves = FALSE, 
                     label_branch_points = FALSE,
                     group_label_size = 5,cell_size=1.) +
        theme(legend.position = "right") +
        guides(color = guide_legend(title = "Cell Type", override.aes = list(size = 3))) +
        ggtitle(paste("n_neighbors =", n_neighbors, "min_dist =", min_dist))
p3 <- plot_cells(cds, 
                 color_cells_by = "pseudotime",
                     label_groups_by_cluster = TRUE,
                     label_leaves = FALSE, 
                     label_branch_points = FALSE,
                     group_label_size = 5,cell_size=1.) +
        theme(legend.position = "right") +
        guides(color = guide_legend(title = "Cell Type", override.aes = list(size = 3))) +
        ggtitle(paste("n_neighbors =", n_neighbors, "min_dist =", min_dist))
p<-p1+p2+p3
      
      # 保存图片
filename <- file.path(deg_dir, "monocle3_umap.pdf")
      ggsave(filename, p, width = 24, height = 8, dpi = 300)

In [ ]:
selected_cells <- colData(cds)$treatment %in% c("M20")
cds_subset1 <- cds[, selected_cells]
selected_cells <- colData(cds)$treatment %in% c("M20_3")
cds_subset2 <- cds[, selected_cells]
selected_cells <- colData(cds)$treatment %in% c("M20_5")
cds_subset3 <- cds[, selected_cells]
p1 <- plot_cells(cds_subset1, 
                     color_cells_by = "anno_sub",
                     label_groups_by_cluster = FALSE,
                     label_leaves = FALSE, 
                     label_branch_points = FALSE,             
                     group_label_size = 5,
                cell_size=0.6) +
        theme(legend.position = "right") +
        guides(color = guide_legend(title = "Cell Type", override.aes = list(size = 3))) +
        ggtitle(paste("n_neighbors =", n_neighbors, "min_dist =", min_dist))
p2 <- plot_cells(cds_subset2, 
                 color_cells_by = "anno_sub",
                     label_groups_by_cluster = TRUE,
                     label_leaves = FALSE, 
                     label_branch_points = FALSE,
                     group_label_size = 5,
                cell_size=0.6) +
        theme(legend.position = "right") +
        guides(color = guide_legend(title = "Cell Type", override.aes = list(size = 3))) +
        ggtitle(paste("n_neighbors =", n_neighbors, "min_dist =", min_dist))
p3 <- plot_cells(cds_subset3, 
                 color_cells_by = "anno_sub",
                     label_groups_by_cluster = TRUE,
                     label_leaves = FALSE, 
                     label_branch_points = FALSE,
                     group_label_size = 5,
                cell_size=0.6) +
        theme(legend.position = "right") +
        guides(color = guide_legend(title = "Cell Type", override.aes = list(size = 3))) +
        ggtitle(paste("n_neighbors =", n_neighbors, "min_dist =", min_dist))
p<-p1+p2+p3
      
      # 保存图片
filename <- file.path(deg_dir, "monocle3_umap_subset.pdf")
      ggsave(filename, p, width = 24, height = 8, dpi = 300)

In [ ]:

library(ggpointdensity)
options(repr.plot.width = 8, repr.plot.height =8)
# Get UMAP coordinates
umap_coords <- reducedDims(cds_subset1)$UMAP
plot_data <- data.frame(UMAP1 = umap_coords[,1], UMAP2 = umap_coords[,2])

# 2D density plot
p <- ggplot(plot_data, aes(x = UMAP1, y = UMAP2)) +
  stat_density_2d(aes(fill = after_stat(density)), 
                  geom = "raster", 
                  contour = FALSE) +
  scale_fill_viridis(option = "inferno") +
  theme_minimal() +
  labs(title = "Cell Density Map") +
  coord_fixed()


filename <- file.path(deg_dir, "density_map_M20.pdf")
ggsave(filename, p, width = 4, height = 8, dpi = 300)

In [ ]:
plot_cells(cds, 
           color_cells_by = "pseudotime",
           label_groups_by_cluster = FALSE,
           label_leaves = FALSE, 
           label_branch_points = FALSE,
           label_principal_points=FALSE,
           show_trajectory_graph=FALSE,
           group_label_size = 5) +
  theme(legend.position = "right") +
  guides(color = guide_legend(title = "Cell Type", override.aes = list(size = 3)))

In [ ]:
colData(cds)$treatment_anno=paste0(colData(cds)$anno_sub,"_",colData(cds)$treatment)

In [ ]:
cds_filtered <- cds[, colData(cds)$anno_sub %in% c("TAM.M2","TAM.prolif","TAM.Fth1","TAM.plac8","TAM.Ccr2","TAM.Ccl")]

In [ ]:
library(stringr)

pseudotime_data <- data.frame(
  pseudotime = pseudotime(cds_filtered),
  anno_sub = colData(cds_filtered)$anno_sub
) %>%
  filter(!is.na(pseudotime)) %>%
  filter(!is.infinite(pseudotime)) %>%
  filter(!is.nan(pseudotime)) 
level_order <- pseudotime_data %>%
  group_by(anno_sub) %>%
  summarise(mean_pt = mean(pseudotime)) %>%
  arrange(mean_pt) %>%
  pull(anno_sub)

pseudotime_data <- pseudotime_data %>%
  mutate(anno_sub = factor(anno_sub, levels = level_order))


summary_pseudotime <- pseudotime_data %>%
  group_by(anno_sub) %>%
  summarise(
    mean_pt = mean(pseudotime, na.rm = TRUE),
    sd_pt = sd(pseudotime, na.rm = TRUE)

  ) %>%
  ungroup()%>%
  arrange(mean_pt)  
summary_pseudotime$anno_sub <- factor(summary_pseudotime$anno_sub, levels = unique(summary_pseudotime$anno_sub))

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 6)

# Bar plot with error bars
p<-ggplot(summary_pseudotime, aes(x = anno_sub, y = mean_pt, fill = anno_sub)) +
  geom_bar(stat = "identity", alpha = 0.8) +
  geom_errorbar(aes(ymin = mean_pt - sd_pt, ymax = mean_pt + sd_pt), 
                width = 0.2, alpha = 0.7) +
  labs(title = "Mean Pseudotime by Cell Subtype (± SD)",
       x = "Cell Subtype", 
       y = "Mean Pseudotime") +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  scale_fill_viridis_d()
filename <- file.path(deg_dir, "pseudotime_bar.pdf")
      ggsave(filename, p, width = 5, height = 6, dpi = 300)

In [ ]:
library(stringr)

pseudotime_data <- data.frame(
  pseudotime = 1-pseudotime(cds),
  anno_sub = colData(cds)$anno_sub,
    treatment=colData(cds)$treatment
) %>%
  filter(!is.na(pseudotime)) %>%
  filter(str_detect(treatment, "M20_5")) 

# Calculate mean pseudotime for each group and order
pseudotime_data <- pseudotime_data %>%
  mutate(anno_sub = factor(anno_sub,
                          levels = pseudotime_data %>%
                            group_by(anno_sub) %>%
                            summarise(mean_pt = mean(pseudotime)) %>%
                            arrange(mean_pt) %>%
                            pull(anno_sub)))
pseudotime_data_reversed <- pseudotime_data %>%
  mutate(anno_sub = factor(anno_sub, levels = rev(levels(anno_sub))))

In [ ]:
ggplot(pseudotime_data_reversed, aes(x = anno_sub, y = pseudotime, fill = anno_sub)) +
  geom_violin(alpha = 0.7) +
  geom_boxplot(width = 0.2, alpha = 0.8) +
  labs(title = "Pseudotime Distribution by Cell Subtype (Ordered by Mean)",
       x = "Cell Subtype", 
       y = "1-Pseudotime") +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  scale_fill_viridis_d()

In [ ]:
saveRDS(cds, file = file.path(deg_dir, "monocle3_cds.rds"))
